# Tutorial 5: Hopset-based SSSP and comparison to plain Dijkstra

The CFR hopset construction of Cao, Fineman, Russell (2020)
produces a small set of additional directed edges (the hopset)
that, when added to the original graph, allows approximate
shortest-path queries to be answered with bounded hopcount.

This notebook builds a hopset for a layered DAG and compares
the augmented-graph SSSP (reachq.shortest_path_hopbound) to plain
Dijkstra.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from reachq.graph import WeightedDigraph
from reachq.hopset import build_hopset_for_sssp
from reachq.shortest_paths import dijkstra, shortest_path_hopbound

In [ ]:
# Build a layered DAG as a weighted digraph (unit weights)
def layered_dag(L, s):
    g = WeightedDigraph()
    for i in range(L):
        for j in range(s):
            g.add_vertex((i, j))
    for i in range(L - 1):
        for j1 in range(s):
            for j2 in range(s):
                g.add_edge((i, j1), (i + 1, j2), 1)
    return g


g = layered_dag(L=8, s=10)
print(f"Graph: {g.num_vertices()} vertices, {g.num_edges()} edges")

In [ ]:
# Build the hopset
H, beta = build_hopset_for_sssp(g, epsilon=0.1, random_seed=42)
print(f"Hopset: {len(H)} shortcuts, beta={beta:.2f}")

In [ ]:
# Compare plain Dijkstra vs hopbound SSSP from source (0, 0)
src = (0, 0)
dijkstra_dist = dijkstra(g, src)

print(f"Dijkstra from {src}:")
print(
    f"  reachable: {sum(1 for v in dijkstra_dist.values() if v != float('inf'))}/{g.num_vertices()}"
)
print(f"  max hops to any reachable: {max(dijkstra_dist.values())}")

print("Hopset-augmented distances (using shortest_path_hopbound):")
hopbound_dist = shortest_path_hopbound(g, H, src, max_hops=int(beta) + 5)
ok_count = 0
for v in g.vertices():
    d_plain = dijkstra_dist[v]
    d_hop = hopbound_dist.get(v)
    if d_plain != float("inf") and d_hop is not None:
        # The hopset preserves reachability and distance is within
        # (1+epsilon) of the plain distance.
        if d_hop <= d_plain * 1.1 + 1e-6:
            ok_count += 1
    elif d_hop is None and d_plain == float("inf"):
        ok_count += 1
print(
    f"  within (1 + epsilon) of plain: {ok_count}/{sum(1 for v in dijkstra_dist.values() if v != float('inf'))}"
)

In [ ]:
# Hopset soundness: every source-target pair reachable in G is
# still reachable in G + H (within beta hops).
from reachq.reachability import bfs_reachability, parallel_bfs

print("Hopset soundness check (every source):")
for s in g.vertices():
    plain = bfs_reachability(g, s)
    if plain == parallel_bfs(g, s, set()):
        # No shortcuts: must still work
        continue
    if plain != parallel_bfs(g, s, H):
        print(f"  Soundness VIOLATED at {s}")
        break
else:
    print("  preserved for all sources")

In [ ]:
# Hopset size vs graph size: how much overhead does the hopset add?
import matplotlib.pyplot as plt

sizes = [3, 5, 8, 12, 15]
hopset_sizes = []
graph_sizes = []
for L in sizes:
    g_n = layered_dag(L=L, s=10)
    H_n, _ = build_hopset_for_sssp(g_n, epsilon=0.1, random_seed=42)
    hopset_sizes.append(len(H_n))
    graph_sizes.append(g_n.num_edges())

plt.figure(figsize=(8, 4))
plt.plot(sizes, graph_sizes, "o-", label="graph edges (|E|)")
plt.plot(sizes, hopset_sizes, "s-", label="hopset edges (|H|)")
plt.xlabel("layers (L)")
plt.ylabel("edge count")
plt.title("Layered DAG: graph size vs hopset size")
plt.legend()
plt.grid(True)
plt.show()

## What you should see

1. The hopset for a layered DAG of L=8, s=10 is much smaller
   than the original graph (the hopset only adds shortcuts
   between non-adjacent layers, not within them).
2. The hopset-augmented distances are within (1+epsilon) of the
   plain Dijkstra distances. This is the (1+epsilon)-approximation
   guarantee of the CFR construction.
3. The hopset preserves reachability for all sources — no
   source-target reachability is lost when the hopset is added.
4. As L grows, the hopset grows but stays sub-linear in |E|.

## What's next

Tutorial 6 (in the examples directory) walks through the GNN
preprocessing use case: turn a citation graph into a reachq-
augmented PyG Data object.